# Advanced AI Data Analyst Demo

## Multi-Genie Orchestration with Parallel Queries and Synthesized Insights

This notebook demonstrates the full multi-agent pipeline for the AI Data Analyst Workshop:

1. **Planning** - PlannerAgent decomposes complex questions into domain-specific sub-queries
2. **Querying** - MultiGenieOrchestrator executes parallel queries across multiple Genie Spaces
3. **Synthesizing** - SynthesizerAgent generates cross-domain insights and correlations
4. **Reporting** - ReportWriter produces Markdown and HTML dashboard outputs

### Key Features
- Parallel query execution with progress tracking
- Graceful degradation when individual spaces fail
- Cross-domain insight generation
- Automated report generation
- Mock mode for demos without real Databricks Genie access

## 1. Setup and Dependencies

In [ ]:
# Install dependencies (run once)
%pip install databricks-sdk>=0.40.0 databricks-langchain>=0.13.0 langgraph>=0.2.0 langchain-core>=0.3.0 pydantic>=2.0.0 python-dotenv>=1.0.0 jinja2>=3.0.0 -q

In [ ]:
# Restart Python to pick up new packages (Databricks)
# dbutils.library.restartPython()

In [ ]:
# Add src to path for imports
import sys
import os

# For Databricks notebooks
if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    # Get the workspace path
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    workspace_path = '/Workspace' + '/'.join(notebook_path.split('/')[:-2])
    if workspace_path not in sys.path:
        sys.path.insert(0, workspace_path)
else:
    # For local development
    project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

## 2. Configuration Widgets

Run the cell below to create interactive configuration widgets.
Use the widgets at the top of the notebook to configure your demo settings.

In [ ]:
# Create Databricks widgets for configuration
# These appear at the top of the notebook for easy access

# Check if running in Databricks
import os
IN_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

if IN_DATABRICKS:
    # Remove existing widgets to avoid duplicates
    try:
        dbutils.widgets.removeAll()
    except:
        pass
    
    # === Genie Space Configuration ===
    dbutils.widgets.text("genie_space_sales_id", "", "1. Sales Space ID")
    dbutils.widgets.text("genie_space_customers_id", "", "2. Customers Space ID")
    dbutils.widgets.text("genie_space_inventory_id", "", "3. Inventory Space ID")
    
    # === Infrastructure ===
    dbutils.widgets.text("warehouse_id", "", "4. Warehouse ID")
    dbutils.widgets.dropdown(
        "model_endpoint",
        "databricks-meta-llama-3-3-70b-instruct",
        ["databricks-meta-llama-3-3-70b-instruct", "databricks-meta-llama-3-1-405b-instruct", "databricks-dbrx-instruct"],
        "5. Model Endpoint"
    )
    
    # === Demo Settings ===
    dbutils.widgets.dropdown("mock_mode", "true", ["true", "false"], "6. Mock Mode")
    dbutils.widgets.dropdown("cache_enabled", "true", ["true", "false"], "7. Cache Enabled")
    dbutils.widgets.dropdown(
        "report_type",
        "Q4 Executive Report",
        ["Q4 Executive Report", "YTD Summary", "Custom Query"],
        "8. Report Type"
    )
    dbutils.widgets.text("custom_query", "", "9. Custom Query (if Report Type = Custom)")
    
    print("Widgets created! Configure settings using the dropdowns at the top of the notebook.")
    print("\nFor Mock Mode demo, leave Space IDs empty and set Mock Mode = true")
else:
    print("Running locally - using environment variables for configuration")
    print("Set: MOCK_MODE=true for demo without Genie access")

In [ ]:
# Read configuration from widgets (Databricks) or environment variables (local)
from src.config import Config, clear_config_cache
from src.agents.multi_genie_orchestrator import GenieSpaceConfig

clear_config_cache()

if IN_DATABRICKS:
    # Read from Databricks widgets
    sales_space_id = dbutils.widgets.get("genie_space_sales_id") or "mock-sales-space"
    customers_space_id = dbutils.widgets.get("genie_space_customers_id") or "mock-customers-space"
    inventory_space_id = dbutils.widgets.get("genie_space_inventory_id") or "mock-inventory-space"
    warehouse_id = dbutils.widgets.get("warehouse_id")
    model_endpoint = dbutils.widgets.get("model_endpoint")
    mock_mode = dbutils.widgets.get("mock_mode") == "true"
    cache_enabled = dbutils.widgets.get("cache_enabled") == "true"
    report_type = dbutils.widgets.get("report_type")
    custom_query = dbutils.widgets.get("custom_query")
else:
    # Read from environment variables (local development)
    sales_space_id = os.getenv("GENIE_SPACE_SALES_ID", "mock-sales-space")
    customers_space_id = os.getenv("GENIE_SPACE_CUSTOMERS_ID", "mock-customers-space")
    inventory_space_id = os.getenv("GENIE_SPACE_INVENTORY_ID", "mock-inventory-space")
    warehouse_id = os.getenv("WAREHOUSE_ID", "")
    model_endpoint = os.getenv("MODEL_ENDPOINT", "databricks-meta-llama-3-3-70b-instruct")
    mock_mode = os.getenv("MOCK_MODE", "true").lower() == "true"
    cache_enabled = os.getenv("CACHE_ENABLED", "true").lower() == "true"
    report_type = os.getenv("REPORT_TYPE", "Q4 Executive Report")
    custom_query = os.getenv("CUSTOM_QUERY", "")

# Create main configuration
config = Config(
    genie_space_id=sales_space_id,
    warehouse_id=warehouse_id,
    model_endpoint=model_endpoint,
    mock_mode=mock_mode,
    cache_enabled=cache_enabled,
)

# Configure Genie Spaces
space_configs = [
    GenieSpaceConfig(
        space_id=sales_space_id,
        name="Sales",
        domain="sales, revenue, orders, transactions",
    ),
    GenieSpaceConfig(
        space_id=customers_space_id,
        name="Customers",
        domain="customers, segments, demographics, retention",
    ),
    GenieSpaceConfig(
        space_id=inventory_space_id,
        name="Inventory",
        domain="inventory, stock, products, warehouse",
    ),
]

# Determine question based on report type
REPORT_QUESTIONS = {
    "Q4 Executive Report": "Generate a Q4 executive report analyzing sales performance, customer trends, and inventory status",
    "YTD Summary": "Provide a year-to-date summary of business performance including revenue trends, customer acquisition, and operational metrics",
    "Custom Query": custom_query or "Analyze the current business state",
}
question = REPORT_QUESTIONS.get(report_type, REPORT_QUESTIONS["Q4 Executive Report"])

# Display configuration summary
print("=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"\nMode: {'MOCK (Demo)' if mock_mode else 'LIVE (Real Genie)'}")
print(f"Cache: {'Enabled' if cache_enabled else 'Disabled'}")
print(f"Model: {model_endpoint}")
print(f"\nGenie Spaces:")
for sc in space_configs:
    status = "mock" if "mock" in sc.space_id else "configured"
    print(f"  - {sc.name} ({sc.domain[:30]}...) [{status}]")
print(f"\nReport Type: {report_type}")
print(f"Question: {question[:80]}{'...' if len(question) > 80 else ''}")

In [ ]:
# === CHECKPOINT: Setup Complete ===
from src.demo import (
    CheckpointManager, create_checkpoint,
    CHECKPOINT_SETUP, display_success, display_demo_marker
)

# Display presenter marker (only visible in presenter mode)
display_demo_marker(5, "Configuration walkthrough and widget explanation", "action")

# Create checkpoint manager and save setup state
checkpoint_manager = CheckpointManager()
create_checkpoint(
    name=CHECKPOINT_SETUP,
    config=config,
    pipeline_state=None,
    space_configs=space_configs,
    manager=checkpoint_manager,
)
display_success("Setup checkpoint saved!", "Configuration captured. You can restore with checkpoint_manager.restore('setup_complete')")

## 3. Demo Mode Selection

This notebook supports two execution modes:

### End-to-End Mode
Run the complete pipeline in a single cell. Best for demos and quick results.

### Step-Through Mode
Execute each stage individually to understand the pipeline flow. Best for learning and debugging.

Skip to **Section 4** for End-to-End mode, or **Section 5** for Step-Through mode.

## 4. End-to-End Mode

Execute the complete multi-agent pipeline in a single cell with progress tracking.

In [ ]:
from IPython.display import display, Markdown, HTML, clear_output

from src.demo import PipelineState, PipelineStage, render_for_notebook, render_progress_html
from src.agents.planner_agent import PlannerAgent
from src.agents.multi_genie_orchestrator import MultiGenieOrchestrator
from src.agents.synthesizer_agent import SynthesizerAgent
from src.agents.report_writer import ReportWriter

# Initialize pipeline state
state = PipelineState()
state.initialize_spaces([sc.name for sc in space_configs])

# Initialize agents with configuration from widgets
planner = PlannerAgent(config, space_configs)
orchestrator = MultiGenieOrchestrator(
    space_configs,
    config,
    progress_callback=state.get_progress_callback(),
)
synthesizer = SynthesizerAgent(config)
report_writer = ReportWriter(config)

print(f"Question: {question}")
print("=" * 80)
print()

try:
    # Stage 1: Planning
    state.start_stage(PipelineStage.PLANNING)
    print("[1/4] Planning - Decomposing question into domain-specific queries...")
    state.plan = planner.decompose(question)
    state.complete_stage(PipelineStage.PLANNING)
    print(f"       Created {len(state.plan.sub_queries)} sub-queries targeting: {', '.join(state.plan.target_spaces)}")
    print()

    # Stage 2: Querying
    state.start_stage(PipelineStage.QUERYING)
    print("[2/4] Querying - Executing parallel queries across Genie Spaces...")
    state.multi_result = orchestrator.query_all(question)
    state.reconcile_from_result(state.multi_result)
    state.complete_stage(PipelineStage.QUERYING)
    
    successful = state.multi_result.successful_results()
    failed = state.multi_result.get_failed_spaces()
    print(f"       Successful: {len(successful)}/{len(space_configs)} spaces")
    if failed:
        print(f"       Failed: {', '.join(failed)}")
    print()

    # Stage 3: Synthesizing
    state.start_stage(PipelineStage.SYNTHESIZING)
    print("[3/4] Synthesizing - Generating cross-domain insights...")
    state.synthesis_result = synthesizer.synthesize(state.multi_result, question)
    state.complete_stage(PipelineStage.SYNTHESIZING)
    print(f"       Generated {len(state.synthesis_result.key_insights)} insights, {len(state.synthesis_result.recommendations)} recommendations")
    print()

    # Stage 4: Reporting
    state.start_stage(PipelineStage.REPORTING)
    print("[4/4] Reporting - Generating Markdown and HTML reports...")
    state.markdown_report = report_writer.generate_markdown(state.synthesis_result, title=report_type)
    state.html_report = report_writer.generate_html(state.synthesis_result, title=report_type)
    state.complete_stage(PipelineStage.REPORTING)
    print("       Reports generated successfully!")
    print()

    # Display final progress with HTML cards
    print("=" * 80)
    cache_stats = orchestrator.get_cache_stats()
    display(HTML(render_progress_html(state, cache_stats=cache_stats)))

except Exception as e:
    state.fail_stage(state.current_stage, str(e))
    print(f"\nPipeline failed: {e}")
    display(Markdown(render_for_notebook(state)))

In [ ]:
# Presenter timing marker (only visible in presenter mode)
from src.demo import display_demo_marker
display_demo_marker(8, "End-to-end pipeline demonstration", "action")

In [ ]:
# Display the generated Markdown report
if state.markdown_report:
    display(Markdown(state.markdown_report))
else:
    print("No report generated. Run the End-to-End cell above first.")

## 5. Step-Through Mode

Execute each pipeline stage individually to understand the data flow.

### 5.1 Initialize Pipeline State and Agents

In [ ]:
from IPython.display import display, Markdown, HTML, clear_output

from src.demo import PipelineState, PipelineStage, render_for_notebook
from src.agents.planner_agent import PlannerAgent
from src.agents.multi_genie_orchestrator import MultiGenieOrchestrator
from src.agents.synthesizer_agent import SynthesizerAgent
from src.agents.report_writer import ReportWriter

# Initialize fresh pipeline state
state = PipelineState()
state.initialize_spaces([sc.name for sc in space_configs])

# Initialize agents
planner = PlannerAgent(config, space_configs)
orchestrator = MultiGenieOrchestrator(
    space_configs,
    config,
    progress_callback=state.get_progress_callback(),
)
synthesizer = SynthesizerAgent(config)
report_writer = ReportWriter(config)

# Define the question
question = "Generate a Q4 executive report analyzing sales performance, customer trends, and inventory status"

print("Pipeline state and agents initialized.")
print(f"\nQuestion: {question}")
print(f"\nConfigured spaces: {[sc.name for sc in space_configs]}")

In [ ]:
# Presenter timing marker (only visible in presenter mode)
from src.demo import display_demo_marker
display_demo_marker(15, "Step-through mode with discussion at each stage", "action")

### 5.2 Planning Stage

The PlannerAgent decomposes the complex question into domain-specific sub-queries.

In [ ]:
# Execute planning stage
state.start_stage(PipelineStage.PLANNING)

try:
    state.plan = planner.decompose(question)
    state.complete_stage(PipelineStage.PLANNING)
    
    print("Planning Complete!")
    print("=" * 60)
    print(f"\nOriginal Question: {state.plan.original_question}")
    print(f"\nTarget Spaces: {state.plan.target_spaces}")
    print(f"\nSub-Queries ({len(state.plan.sub_queries)}):")
    for i, sq in enumerate(state.plan.sub_queries, 1):
        print(f"  {i}. [{sq.target_space}] {sq.query}")
    print(f"\nSynthesis Instructions: {state.plan.synthesis_instructions}")
    
except Exception as e:
    state.fail_stage(PipelineStage.PLANNING, str(e))
    print(f"Planning failed: {e}")

# Show progress
print("\n" + "=" * 60)
display(Markdown(render_for_notebook(state, include_timing=True)))

In [ ]:
# === CHECKPOINT: Planning Complete ===
from src.demo import CHECKPOINT_BASIC_QUERY, create_checkpoint, display_success

create_checkpoint(
    name=CHECKPOINT_BASIC_QUERY,
    config=config,
    pipeline_state=state,
    space_configs=space_configs,
    extras={"plan": state.plan.original_question if state.plan else ""},
    manager=checkpoint_manager,
)
display_success("Planning checkpoint saved!", f"Plan created with {len(state.plan.sub_queries) if state.plan else 0} sub-queries.")

### 5.3 Querying Stage

The MultiGenieOrchestrator executes parallel queries across all Genie Spaces.

### Challenge: Change the Report Type

Customize the pipeline by changing the report type or adding a new Genie Space.

In [ ]:
# === CHALLENGE: Change Report Type ===
from src.demo import ADVANCED_CHALLENGES, get_challenge_runner, display_solution

# Get the challenge and runner
challenge = ADVANCED_CHALLENGES["change_report_type"]
runner = get_challenge_runner()

# Display the challenge instructions
runner.show_challenge(challenge)

# Show solution (only visible in presenter mode)
display_solution(
    challenge.solution_code,
    "Change report_type to 'YTD Summary' or use 'Custom Query' with your own question."
)

In [ ]:
# YOUR CODE HERE: Modify the report type
# Option 1: Change report_type to "YTD Summary"
# Option 2: Use "Custom Query" with custom_query set to your question

# After making changes, re-run the configuration and pipeline cells above
# Then pass state.markdown_report or state.synthesis_result to validation below

# Example: Change report_type in the configuration cell to:
# report_type = "YTD Summary"
# Then re-run from configuration through reporting

# For validation, pass your result:
# from src.demo import run_challenge
# run_challenge(challenge, state.markdown_report)  # or state.synthesis_result

In [ ]:
# Execute querying stage
state.start_stage(PipelineStage.QUERYING)

try:
    state.multi_result = orchestrator.query_all(question)
    state.reconcile_from_result(state.multi_result)
    state.complete_stage(PipelineStage.QUERYING)
    
    print("Querying Complete!")
    print("=" * 60)
    
    # Show results summary
    successful = state.multi_result.successful_results()
    failed = state.multi_result.get_failed_spaces()
    
    print(f"\nSuccessful Queries: {len(successful)}/{len(space_configs)}")
    for name, result in successful.items():
        meta = state.multi_result.metadata.get(name)
        timing = f" ({meta.query_time_seconds:.2f}s)" if meta else ""
        row_count = len(result.data) if result.data else 0
        print(f"  - {name}: {row_count} rows{timing}")
    
    if failed:
        print(f"\nFailed Queries: {len(failed)}")
        for name in failed:
            result = state.multi_result.results.get(name)
            error = result.error if result else "Unknown error"
            print(f"  - {name}: {error}")
    
    # Show cache stats if available
    cache_stats = orchestrator.get_cache_stats()
    if cache_stats:
        print(f"\nCache Stats: {cache_stats.get('hits', 0)} hits, {cache_stats.get('misses', 0)} misses")
        
except Exception as e:
    state.fail_stage(PipelineStage.QUERYING, str(e))
    print(f"Querying failed: {e}")

# Show progress
print("\n" + "=" * 60)
display(Markdown(render_for_notebook(state, include_timing=True)))

In [ ]:
# === CHECKPOINT: Querying Complete ===
from src.demo import CHECKPOINT_MULTI_AGENT, create_checkpoint, display_success

successful_count = len(state.multi_result.successful_results()) if state.multi_result else 0
create_checkpoint(
    name=CHECKPOINT_MULTI_AGENT,
    config=config,
    pipeline_state=state,
    space_configs=space_configs,
    extras={"successful_spaces": successful_count},
    manager=checkpoint_manager,
)
display_success("Query checkpoint saved!", f"Successfully queried {successful_count} Genie Spaces.")

In [ ]:
# Display detailed query results
if state.multi_result:
    display(Markdown(state.multi_result.to_combined_markdown(max_rows_per_space=5)))
else:
    print("No query results available. Run the querying stage first.")

### 5.4 Synthesizing Stage

The SynthesizerAgent combines results to generate cross-domain insights.

In [ ]:
# Execute synthesizing stage
state.start_stage(PipelineStage.SYNTHESIZING)

try:
    state.synthesis_result = synthesizer.synthesize(state.multi_result, question)
    state.complete_stage(PipelineStage.SYNTHESIZING)
    
    print("Synthesis Complete!")
    print("=" * 60)
    
    result = state.synthesis_result
    print(f"\nDomains Analyzed: {', '.join(result.domains_analyzed)}")
    
    if result.domains_unavailable:
        print(f"Domains Unavailable: {', '.join(result.domains_unavailable)}")
    
    print(f"\nKey Insights: {len(result.key_insights)}")
    for i, insight in enumerate(result.key_insights, 1):
        print(f"  {i}. [{insight.importance.upper()}] {insight.insight}")
    
    print(f"\nCorrelations: {len(result.cross_domain_correlations)}")
    for corr in result.cross_domain_correlations:
        print(f"  - {corr.description}")
    
    print(f"\nAnomalies: {len(result.anomalies)}")
    for anomaly in result.anomalies:
        print(f"  - [{anomaly.severity.upper()}] {anomaly.description}")
    
    print(f"\nRecommendations: {len(result.recommendations)}")
    for i, rec in enumerate(result.recommendations, 1):
        print(f"  {i}. {rec}")
        
except Exception as e:
    state.fail_stage(PipelineStage.SYNTHESIZING, str(e))
    print(f"Synthesis failed: {e}")

# Show progress
print("\n" + "=" * 60)
display(Markdown(render_for_notebook(state, include_timing=True)))

### 5.5 Reporting Stage

The ReportWriter generates formatted Markdown and HTML reports.

In [ ]:
# Execute reporting stage
state.start_stage(PipelineStage.REPORTING)

try:
    state.markdown_report = report_writer.generate_markdown(
        state.synthesis_result,
        title="Q4 Executive Report"
    )
    state.html_report = report_writer.generate_html(
        state.synthesis_result,
        title="Q4 Executive Dashboard"
    )
    state.complete_stage(PipelineStage.REPORTING)
    
    print("Reporting Complete!")
    print("=" * 60)
    print(f"\nMarkdown report: {len(state.markdown_report)} characters")
    print(f"HTML report: {len(state.html_report)} characters")
    
except Exception as e:
    state.fail_stage(PipelineStage.REPORTING, str(e))
    print(f"Reporting failed: {e}")

# Show final progress
print("\n" + "=" * 60)
display(Markdown(render_for_notebook(state, include_timing=True)))

In [ ]:
# === CHECKPOINT: Report Generated ===
from src.demo import CHECKPOINT_REPORT_GENERATED, create_checkpoint, display_success

create_checkpoint(
    name=CHECKPOINT_REPORT_GENERATED,
    config=config,
    pipeline_state=state,
    space_configs=space_configs,
    extras={
        "markdown_length": len(state.markdown_report) if state.markdown_report else 0,
        "html_length": len(state.html_report) if state.html_report else 0,
    },
    manager=checkpoint_manager,
)
display_success("Report checkpoint saved!", "Full pipeline complete. Reports generated successfully.")

## 6. Results Display

View the generated reports.

### HTML Progress Visualization (Phase 2)

Display pipeline progress using HTML progress cards instead of markdown tables.

In [ ]:
# Display HTML progress cards (Phase 2 visualization)
from src.demo import render_progress_html

if state.total_duration_seconds is not None:
    # Get cache stats for display
    cache_stats = orchestrator.get_cache_stats() if 'orchestrator' in dir() else None
    
    # Render HTML progress visualization
    html_progress = render_progress_html(state, cache_stats=cache_stats)
    display(HTML(html_progress))
else:
    print("Pipeline not yet executed. Run End-to-End or Step-Through mode first.")

In [ ]:
# Display the Markdown report
if state.markdown_report:
    display(Markdown(state.markdown_report))
else:
    print("No Markdown report available. Complete the pipeline first.")

In [ ]:
# Display the HTML dashboard (if supported by the notebook environment)
if state.html_report:
    # For Databricks notebooks or Jupyter with HTML support
    display(HTML(state.html_report))
else:
    print("No HTML report available. Complete the pipeline first.")

In [ ]:
# Alternative: Display synthesis result directly using its built-in markdown formatter
if state.synthesis_result:
    display(Markdown(state.synthesis_result.to_markdown()))
else:
    print("No synthesis result available.")

## 7. Cleanup

Reset pipeline state and clear caches for a fresh run.

In [ ]:
# Reset pipeline state
state.reset()
print("Pipeline state reset.")

# Reset orchestrator conversations
orchestrator.reset_all_conversations()
print("Orchestrator conversations reset.")

# Invalidate query cache
invalidated = orchestrator.invalidate_cache()
print(f"Cache invalidated ({invalidated} entries cleared).")

# Clear config cache
clear_config_cache()
print("Configuration cache cleared.")

print("\nReady for a fresh run!")

---

## Architecture Summary

```
                    User Question
                          |
                          v
                 +----------------+
                 | PlannerAgent   |  Decompose question into
                 | (LLM-powered)  |  domain-specific sub-queries
                 +----------------+
                          |
                          v
          +-------------------------------+
          |    MultiGenieOrchestrator     |  Parallel execution
          |                               |  with progress tracking
          +-------------------------------+
                /         |         \
               v          v          v
        +--------+  +----------+  +-----------+
        | Sales  |  | Customer |  | Inventory |
        | Genie  |  | Genie    |  | Genie     |
        +--------+  +----------+  +-----------+
               \          |          /
                v         v         v
          +-------------------------------+
          |     SynthesizerAgent          |  Cross-domain insights
          |     (LLM-powered)             |  and recommendations
          +-------------------------------+
                          |
                          v
                 +----------------+
                 | ReportWriter   |  Markdown and HTML
                 | (Jinja2)       |  dashboard generation
                 +----------------+
                          |
                          v
                  Final Reports
```

### Key Components

| Component | Purpose | Technology |
|-----------|---------|------------|
| PlannerAgent | Query decomposition | ChatDatabricks LLM |
| MultiGenieOrchestrator | Parallel query execution | ThreadPoolExecutor |
| PipelineState | Progress tracking | Thread-safe callbacks |
| SynthesizerAgent | Cross-domain analysis | ChatDatabricks LLM |
| ReportWriter | Report generation | Jinja2 templates |

---

## Next Steps

### Continue Learning

- **Build Your Own Agent**: Hands-on workshop for building LangGraph agents
  - [03_build_your_agent.ipynb](./03_build_your_agent.ipynb)

- **Basic Demo**: See the simple multi-agent system in action
  - [demo.ipynb](./demo.ipynb)